# 18 · Real Cisco and SEC Retrieval & Safety Validation

**Run All** after notebook 17 (this notebook reruns 17 safely). Tests all seven imported PDFs, real-source title retrieval, project isolation, and the fact that a proxy filing must not be described as a Form 10-K. These are smoke checks for **document discovery**, not expert validation of advice. The scores from TF-IDF/BM25 are relevance heuristics, not probabilities.

If one retrieval test fails after additional files are indexed, inspect the retrieved evidence and adjust query context rather than fabricating an answer.

In [1]:
%run 03_evidence_recommendations.ipynb
%run 17_official_pdf_import.ipynb
from time import perf_counter
from pprint import pprint
# Case queries are grounded in real cover/section titles rather than invented model-part equivalences.
CHECKS=[
 {'domain':'technology','query':'Cisco Catalyst 8300 Series Edge Platforms optics transceivers modules redundant power supplies product specifications', 'expected':'datasheet-c78-744088.pdf'},
 {'domain':'technology','query':'Cisco 4000 Family Integrated Services Router ISR4000 supported interfaces modules branch in a box', 'expected':'data_sheet-c78-732542.pdf'},
 {'domain':'technology','query':'Cisco ASR 1000 Series Aggregation Services Routers ESP route processor ordering specifications', 'expected':'datasheet-c78-731632.pdf'},
 {'domain':'technology','query':'Cisco Catalyst 8000V Edge Software cloud virtual router SD-WAN features and licensing', 'expected':'catalyst-8000v-edge-software-ds.pdf'},
 {'domain':'sec','query':'Agilent 2025 proxy annual shareholder meeting March 13 2025 and director election', 'expected':'agilent.pdf'},
 {'domain':'sec','query':'Agilent Technologies March 18 2026 annual shareholder meeting proxy statement director election', 'expected':'agient2.pdf'},
 {'domain':'sec','query':'KLA Corporation 2024 proxy Annual Meeting November 6 2024 board governance', 'expected':'agilent3.pdf'},
]
summary=[]
for check in CHECKS:
    started=perf_counter()
    hits=hybrid_search(PROJECT_ID,check['query'],check['domain'],top_k=5,exclude_synthetic=True)
    rank=next((i+1 for i,h in enumerate(hits) if h['filename']==check['expected']),None)
    valid=all(citation_exists(PROJECT_ID,h['chunk_id']) for h in hits)
    summary.append({'domain':check['domain'],'expected':check['expected'],
                    'rank_at_5':rank,'citations_valid':valid,
                    'latency_ms':round((perf_counter()-started)*1000,1),
                    'first_hits':[{'filename':h['filename'],'citation':h['citation'],'score':h['score']} for h in hits[:3]]})
    print(check['domain'],check['expected'],'rank=',rank,'citation_valid=',valid)
    for h in hits[:2]:print('   ',h['citation'],'score=',h['score'])
print('7-case title recall@5:',sum(x['rank_at_5'] is not None for x in summary),'/',len(CHECKS))
assert all(x['citations_valid'] for x in summary),'A returned citation did not resolve'
assert sum(x['rank_at_5'] is not None for x in summary)>=5, 'Retrieval regression: fewer than 5 of 7 expected sources in top 5' 

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Document ingestion functions ready
Offline hybrid retrieval ready
Optional native pgvector adapter defined; credentials/database not required for local demo
Evidence-safe recommender ready
Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Document ingestion functions ready
Found 7 official PDF records
{
  "documents_processed": 7,
  "newly_indexed": 0,
  "new_chunks": 0,
  "already_indexed": 7,
  "by_domain": {
    "technology": 4,
    "sec": 3
  },
  "notice": "Agilent SEC filings here are DEF 14A proxies; 2025 10-K is still not supplied."
}
technology  datasheet-c78-744088.pdf              24 pp  already indexed  Cisco (technical_datasheet)
technology  data_sheet-c78-732542.pdf             28 pp  already indexed  Cisco (technical_datasheet)
technology  datasheet-c78-731632.pdf              31 pp  already indexed  Cisco (technical_da

In [2]:
# Verify that the raw metadata in the persistent index matches actual SEC document type and organization.
with db_connect() as db:
    db_docs={r['filename']:dict(r) for r in db.execute(
        "SELECT filename,domain,source_type,source_url FROM documents WHERE project_id=? AND filename IN ("+
        ','.join('?' for _ in official_docs)+")",[PROJECT_ID]+[m['filename'] for m in official_docs])}
for meta in official_docs:
    doc=db_docs[meta['filename']]
    assert doc['domain']==meta['domain']
    assert doc['source_url']==meta['source_url']
    if meta['document_type']=='sec_def14a_proxy':
        assert doc['source_type']=='sec-def14a-proxy'
        assert '10k' not in doc['source_type'].lower()
    else:assert doc['source_type']=='official-cisco-datasheet'

other='88888888-8888-4888-8888-888888888888'
assert hybrid_search(other,'Cisco Catalyst 8300 optics',domain='technology',top_k=3)==[]
assert hybrid_search(other,'Agilent proxy statement',domain='sec',top_k=3)==[]
print('PASS: all seven provenance records and cross-project isolation')

PASS: all seven provenance records and cross-project isolation


In [3]:
# Explicit evidence labeling for Group 1 and Group 3: NEVER claim a 2025 10-K is present.
source_inventory={m['filename']:{'company':m['organization'],'document_type':m['document_type'],
                                'year':m['year'],'domain':m['domain'],'source_url':m['source_url']}
                  for m in official_docs}
proxy_count=sum(m['document_type']=='sec_def14a_proxy' for m in official_docs)
ten_k_count=sum(m['document_type']=='sec_form_10k' for m in official_docs)
assert proxy_count==3 and ten_k_count==0
print(json.dumps({'technical_datasheets':4,'sec_proxy_statements':proxy_count,
                  'sec_2025_form_10k_uploaded':False,
                  'imported_manufacturers':['Cisco','Agilent Technologies','KLA Corporation'],
                  'warning':'Proxy statements do not substitute for 2025 10-K annual reports.'},indent=2))
print('PASS: SEC report classification and disclosure warnings')

{
  "technical_datasheets": 4,
  "sec_proxy_statements": 3,
  "sec_2025_form_10k_uploaded": false,
  "imported_manufacturers": [
    "Cisco",
    "Agilent Technologies",
    "KLA Corporation"
  ],
  "warning": "Proxy statements do not substitute for 2025 10-K annual reports."
}
PASS: SEC report classification and disclosure warnings


### What Group 1 / Group 3 should now call

- `agent_knowledge_tool(project_id, 'technology', 'Catalyst 8300 optics transceivers', 5)` retrieves genuine Cisco datasheet excerpts and cited page numbers.
- `agent_knowledge_tool(project_id, 'sec', 'Agilent March 2025 shareholder meeting', 5)` retrieves genuine **proxy** passages; summarize only what the cited documents support.
- API route: `POST /api/v1/projects/{project_id}/knowledge/search` body `{"question":"Cisco Catalyst 8300 supported modules","domain":"technology","top_k":5}`.
- Existing recommendation endpoint: `POST /api/v1/projects/{project_id}/recommendations/rank` must **abstain** from recommending specific replacement SKUs until exact model and co-occurring compatibility evidence are documented.

**Do not run notebook 06's fictional examples as real Cisco claims.** The new notebooks use real uploaded Cisco and SEC documents.